In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json

# ==========================================
# Event Hub Connection
# ==========================================

connectionString = (
    "Endpoint=sb://eh-social-media.servicebus.windows.net/;"
    "SharedAccessKeyName=sudheer;"
    "SharedAccessKey=1IhrynvVGzNbhQLyvqMUxuT7QdTRHRh57+AEhJK7+eQ=;"
    "EntityPath=user-metadata-hub"
)

ehConf = {}

ehConf["eventhubs.connectionString"] = (
    sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(connectionString)
)

ehConf["eventhubs.consumerGroup"] = "$Default"

startingEventPosition = {
    "offset": "-1",
    "seqNo": -1,
    "enqueuedTime": None,
    "isInclusive": True
}

ehConf["eventhubs.startingPosition"] = json.dumps(startingEventPosition)
ehConf["eventhubs.maxEventsPerTrigger"] = "5000"

# ==========================================
# Read Stream
# ==========================================

rawDF = (
    spark.readStream
         .format("eventhubs")
         .options(**ehConf)
         .load()
)

eventDF = rawDF.selectExpr(
    "CAST(body AS STRING) AS message"
)

# ==========================================
# Schema
# ==========================================

schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("country", StringType(), True),
    StructField("topic_category", StringType(), True),
    StructField("account_created_date", StringType(), True),
    StructField("followers_count", DoubleType(), True),
    StructField("following_count", DoubleType(), True),
    StructField("likes_count", DoubleType(), True),
    StructField("shares_count", DoubleType(), True),
    StructField("posts_count", DoubleType(), True),
    StructField("verified", StringType(), True)
])

# ==========================================
# Parse JSON
# ==========================================

parsedDF = (
    eventDF
        .select(from_json(col("message"), schema).alias("data"))
        .select("data.*")
)

# ==========================================
# Error Records
# ==========================================

errorDF = parsedDF.filter(col("user_id").isNull())

errorQuery = (
    errorDF.writeStream
        .trigger(availableNow=True)
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "abfss://raw@socialmedia1.dfs.core.windows.net/checkpoints/error_bronze_user_metadata"
        )
        .toTable("socialmedia_databricks.error.error_bronze_user_metadata")
)

# ==========================================
# Bronze Data
# ==========================================

bronzeDF = (
    parsedDF
    .filter(col("user_id").isNotNull())

    .withColumn(
        "account_created_date",
        to_timestamp(
            col("account_created_date"),
            "dd-MM-yyyy HH:mm"
        )
    )

    .withColumn("bronze_load_time", current_timestamp())
    .withColumn("pipeline_name", lit("Bronze_User_Metadata"))
    .withColumn("source_system", lit("Azure Event Hub"))
    .withColumn("ingestion_date", current_date())

    .withWatermark("account_created_date", "10 minutes")
)

# ==========================================
# Write Bronze Table
# ==========================================

bronzeQuery = (
    bronzeDF.writeStream
        .trigger(availableNow=True)
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "abfss://raw@socialmedia1.dfs.core.windows.net/checkpoints/bronze_user_metadata"
        )
        .option("mergeSchema", "true")
        .toTable("socialmedia_databricks.bronze.bronze_user_metadata")
)

bronzeQuery.awaitTermination()
errorQuery.awaitTermination()

print("Bronze User Metadata Loaded Successfully")

Bronze User Metadata Loaded Successfully


In [0]:
%sql
select * from socialmedia_databricks.bronze.bronze_user_metadata;

user_id,country,topic_category,account_created_date,followers_count,following_count,likes_count,shares_count,posts_count,verified,bronze_load_time,pipeline_name,source_system,ingestion_date
9001.0,USA,finance,null,36196.0,4047.0,1118.0,374.0,854.0,Y,2026-07-09T07:06:15.296Z,Bronze_User_Metadata,Azure Event Hub,2026-07-09
4869.0,USA,politics,2025-01-19T21:06:00Z,74415.0,2211.0,1641.0,124.0,835.0,"""NaN""",2026-07-09T07:06:15.296Z,Bronze_User_Metadata,Azure Event Hub,2026-07-09
3019.0,UK,sports,2025-01-21T07:37:00Z,NaN,2234.0,620.0,311.0,61.0,Y,2026-07-09T07:06:15.296Z,Bronze_User_Metadata,Azure Event Hub,2026-07-09
3107.0,Canada,finance,2025-01-04T19:32:00Z,14433.0,3954.0,1442.0,274.0,306.0,Y,2026-07-09T07:06:15.296Z,Bronze_User_Metadata,Azure Event Hub,2026-07-09
3494.0,UK,finance,2025-01-15T16:07:00Z,59970.0,3207.0,1535.0,88.0,955.0,N,2026-07-09T07:06:15.296Z,Bronze_User_Metadata,Azure Event Hub,2026-07-09
6014.0,UK,finance,2025-01-19T10:14:00Z,99007.0,3567.0,991.0,341.0,755.0,N,2026-07-09T07:06:15.296Z,Bronze_User_Metadata,Azure Event Hub,2026-07-09
3918.0,Canada,politics,2025-01-09T17:26:00Z,32008.0,4331.0,1711.0,130.0,137.0,N,2026-07-09T07:06:15.296Z,Bronze_User_Metadata,Azure Event Hub,2026-07-09
1965.0,Germany,sports,2025-01-12T22:16:00Z,73191.0,3273.0,1851.0,194.0,70.0,N,2026-07-09T07:06:15.296Z,Bronze_User_Metadata,Azure Event Hub,2026-07-09
7844.0,India,sports,2025-01-14T17:59:00Z,85624.0,NaN,37.0,187.0,315.0,Y,2026-07-09T07:06:15.296Z,Bronze_User_Metadata,Azure Event Hub,2026-07-09
5994.0,UK,politics,2025-01-01T01:52:00Z,11517.0,2003.0,1372.0,102.0,539.0,Y,2026-07-09T07:06:15.296Z,Bronze_User_Metadata,Azure Event Hub,2026-07-09
